<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/K21_KhonaPlus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/parashar21/main/images/khona21-v0.png) <br>


---



[Prithwis Mukerjee](http://www.yantrajaal.com) ||
Website - [Parashar21](https://parashar21.blogspot.com)<br>

# Khona21 — Interpretive Jyotiṣa

**Khona21** is an experimental AI-assisted Jyotiṣa interpretation platform, developed as a companion to **Parashar21**, which performs the astronomical calculations underlying traditional Indian astrology.

Named after the legendary Bengali astrologer Khona, the platform combines precomputed Rashi and Navamsha charts, Vimshottari Dasha schedules and planetary transits (Gochar) with a curated corpus of Jyotiṣa literature, currently comprising works by **James Braha** and **Harihar Majumdar**.

Rather than relying on hard-coded interpretive rules, Khona21 employs **source-grounded, multipass LLM reasoning** to examine planetary configurations, yogas, cancellations, exceptions and conflicting interpretations. An initial assessment establishes the horoscope's principal characteristics, while subsequent consultations investigate specific questions using the original reference texts and relevant astrological evidence.

This notebook, **k21_KhonaPlus**, implements the consultation workflow, integrating deterministic calculations from Parashar21 with the interpretive capabilities of modern Large Language Models.

**Khona21 explores the convergence of traditional Jyotiṣa knowledge and modern artificial intelligence, combining computational precision with source-grounded reasoning to produce systematic, transparent and auditable astrological interpretations.**

In [1]:
!pip -q install pypandoc
!pip -q install python-docx
from IPython.display import display, Markdown

!wget -O k21core.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21core.py
!wget -O k21fileops.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21fileops.py
from k21core import *
from k21fileops import *

In [2]:
# ------------------------------------------------
# required for multiple imports of the same module
# ------------------------------------------------
#import k21core
#import importlib

#importlib.reload(k21core)
from k21core import *

In [3]:
# ------------------------------------------------
# Set Parameters
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
model = "gpt-6-luna"
#model = "gpt-6-astra"

id = "TG"

###Access Control
This notebook accesses Google Drive to access files and uses the OpenAI API. Hence it can be used ONLY from the userid of the creator. Others will have to create their own access paths

In [4]:
# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF


In [5]:
!wget -O braha.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/BR_Curated_22.txt"
!wget -O harihar.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/HM_Clean_07.txt"
!wget -O role.txt -q https://raw.githubusercontent.com/prithwis/parashar21/main/RoleFiles/khona261027.txt

BRAHA   = "/content/braha.txt"
HARIHAR = "/content/harihar.txt"
ROLE    = "/content/role.txt"

In [6]:
#
# Following files from Parashar21 need to be uploaded here
#

CHART = "/content/Chart.txt"
NAVAMSA = "/content/Navamsa_Chart.txt"
DASHA = "/content/Dasha.txt"
GOCHAR = "/content/Gochar_Chart.txt"

In [7]:
# ------------------------------------------------
# Define the ROLE
# ------------------------------------------------
#RoleFile = f"{K21_HOME}/Roles/khona261027.txt"
khona21 = readText(ROLE)

/content/role.txt | 5,554 characters


In [8]:
# ------------------------------------------------
# Set the Context
# ------------------------------------------------
#BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)
BaseCONTEXT = getContext(
    BRAHA=BRAHA,
    HARIHAR=HARIHAR,
    CHART=CHART,
    NAVAMSA=NAVAMSA,
    DASHA=DASHA
)

In [9]:
# ------------------------------------------------
# FIRST PASS
#
# ------------------------------------------------

FirstPass_Instruction = """
FIRST PASS — FOUNDATIONAL ASSESSMENT

Prepare a concise, high-level assessment of this horoscope.

Present an overall picture of the native's principal
strengths, weaknesses and significant planetary configurations.

Identify the most important favourable and unfavourable yogas,
sign exchanges, cancellations and mitigating factors,
to the extent that they are evident from the supplied data
and reference texts.

Begin with a simple-language summary, followed by a brief
astrological explanation with relevant source references.

This is an initial orientation, not an exhaustive audit.
Focus on the dominant features of the horoscope rather than
attempting to catalogue every planetary condition or yoga.

Distinguish chart facts from interpretations, and acknowledge
important contradictions or uncertainties.

Do not invent configurations or use rules unsupported by
the supplied reference texts.

Do not make predictions about specific future events.
"""

In [10]:
# ------------------------------------------------
# Assemble the first pass query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(BaseCONTEXT, FirstPass_Instruction)

FirstPass = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)


OPENAI | gpt-6-luna | Effort: medium | In: 222,416 | Cached: 0 | Out: 7,259 | Reasoning: 5,696 | Time: 104.3s


In [ ]:
#display(Markdown(FirstPass))
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
#MD_FILE = saveAnswer(id, model, FirstPass)

In [11]:
def augmentContext(baseContext, firstPass):
    """
    Augment the original context with the FirstPass horoscope assessment.
    Returns a new context without modifying the original.
    """
    return (
        baseContext
        + "\n\n<FOUNDATIONAL_ASSESSMENT>\n"
        + firstPass
        + "\n</FOUNDATIONAL_ASSESSMENT>\n"
    )

In [12]:
CONTEXT = augContext(BaseCONTEXT, FIRSTPASS=FirstPass)

In [13]:
# ------------------------------------------------
# Ask your question / define the task
# This and subsequent cells can be run multiple times
# ------------------------------------------------
CurrentQuestion = """
Native has developed a new technique to simulate geopolitical scenarios.
He has offered this technique to a few possible magazines, portals and think tanks.
What is the likelihood of this technique being accepted or appreciated?
"""

In [14]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(CONTEXT, CurrentQuestion)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6-luna | Effort: medium | In: 223,852 | Cached: 1,098 | Out: 5,782 | Reasoning: 4,660 | Time: 87.8s


In [15]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(id, model, answer)
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, "TopGun", CurrentQuestion, provider,model)

Saved: /content/K21_TG_gpt-6-luna_261028_105237.md
Saved: /content/K21_TG_gpt-6-luna_261028_105237.docx


In [16]:
Gochar = readText(GOCHAR)

/content/Gochar_Chart.txt | 4,870 characters


In [17]:
CONTEXT = augContext(
    BaseCONTEXT,
    FIRSTPASS=FirstPass,
    GOCHAR=Gochar
)

In [18]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(CONTEXT, CurrentQuestion)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6-luna | Effort: medium | In: 225,332 | Cached: 1,098 | Out: 7,725 | Reasoning: 6,108 | Time: 110.2s


In [19]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(id, model, answer)
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, "TopGun", CurrentQuestion, provider,model)

Saved: /content/K21_TG_gpt-6-luna_261028_105554.md
Saved: /content/K21_TG_gpt-6-luna_261028_105554.docx


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)